In [1]:
!pip install -q gradio transformers torch accelerate

In [2]:
"""
HTML Quick Styler – AI-Powered HTML Page Generator
SmartBridge Use-Case 2
Tech: Python + Gradio + IBM Granite 3.3 2B Instruct (GPU) or Template Engine (CPU)
"""

import gradio as gr
import torch

# ─────────────────────────────────────────────────────────────────────────────
# MODE DETECTION
# ─────────────────────────────────────────────────────────────────────────────
USE_GPU_MODEL = torch.cuda.is_available()   # True on Colab GPU / local GPU
MODEL_LOADED  = False
tokenizer     = None
model         = None

if USE_GPU_MODEL:
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM
        MODEL_ID  = "ibm-granite/granite-3.3-2b-instruct"
        print(f"GPU detected. Loading {MODEL_ID} ...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        model     = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True,
        )
        model.eval()
        MODEL_LOADED = True
        print("IBM Granite model loaded successfully!")
    except Exception as e:
        print(f"Model load failed: {e}. Falling back to Template Engine.")
        USE_GPU_MODEL = False
else:
    print("No GPU detected. Running in fast Template Engine mode.")
    print("For full IBM Granite AI generation, run on Google Colab with GPU.")


# ─────────────────────────────────────────────────────────────────────────────
# TEMPLATE ENGINE (CPU / no-model fallback)
# ─────────────────────────────────────────────────────────────────────────────
COLOR_MAP = {
    "Indigo / Purple": {"primary": "#6366f1", "light": "rgba(99,102,241,.1)"},
    "Orange / Warm":   {"primary": "#f0803c", "light": "rgba(240,128,60,.1)"},
    "Blue / Ocean":    {"primary": "#3b82f6", "light": "rgba(59,130,246,.1)"},
    "Green / Emerald": {"primary": "#10b981", "light": "rgba(16,185,129,.1)"},
    "Pink / Rose":     {"primary": "#ec4899", "light": "rgba(236,72,153,.1)"},
    "Dark / Midnight": {"primary": "#8b5cf6", "light": "rgba(139,92,246,.1)"},
    "Red / Crimson":   {"primary": "#ef4444", "light": "rgba(239,68,68,.1)"},
    "Teal / Cyan":     {"primary": "#06b6d4", "light": "rgba(6,182,212,.1)"},
}

SERVICE_MAP = {
    "Portfolio":           [("🎨","UI/UX Design","Pixel-perfect interfaces built with Figma and design thinking."),
                            ("💻","Web Development","Fast, responsive sites with clean HTML, CSS & JavaScript."),
                            ("📱","Mobile Design","Intuitive mobile-first experiences users love.")],
    "Business / Corporate":[("🚀","Digital Strategy","Data-driven plans to grow your online presence."),
                            ("💡","Brand Identity","Logos and brand systems that stand out."),
                            ("📊","SEO & Analytics","Rank higher, convert better, grow faster.")],
    "Landing Page":        [("⚡","Lightning Fast","Sub-second load times that convert visitors."),
                            ("🔒","Secure by Default","Enterprise-grade security from day one."),
                            ("🤝","24/7 Support","Always available to help you grow.")],
    "Education / School":  [("🎓","Expert Instructors","Learn from industry professionals."),
                            ("📚","Rich Curriculum","Job-ready skills built step by step."),
                            ("🏆","Certificates","Globally recognised credentials.")],
    "Blog":                [("✍️","In-Depth Guides","Comprehensive tutorials on trending topics."),
                            ("🔥","Fresh Content","New posts every week."),
                            ("💬","Community","Join thousands of engaged readers.")],
    "E-Commerce":          [("🛒","Curated Products","Handpicked for quality and value."),
                            ("🚚","Fast Delivery","Same-day options across major cities."),
                            ("↩️","Easy Returns","30-day hassle-free returns.")],
}

HERO_MAP = {
    "Portfolio":           ("Hi, I'm {brand}", "Creative designer crafting extraordinary digital experiences."),
    "Business / Corporate":("Grow with {brand}", "Digital solutions that move your business forward."),
    "Landing Page":        ("Build Faster with {brand}", "The all-in-one platform to supercharge your workflow."),
    "Education / School":  ("Learn Smarter at {brand}", "Build skills that last a lifetime with expert guidance."),
    "Blog":                ("Ideas That Inspire", "Stories, insights, and ideas worth sharing."),
    "E-Commerce":          ("Shop the Best at {brand}", "Curated products that elevate every day."),
}

STYLE_FONTS = {
    "Modern":       "'Inter', sans-serif",
    "Minimal":      "'Inter', sans-serif",
    "Creative":     "'Georgia', serif",
    "Professional": "'Segoe UI', sans-serif",
    "Bold & Dark":  "'Georgia', serif",
    "Elegant":      "'Georgia', serif",
}

def template_generate(title, desc, wtype, style, color_theme, cta, brand):
    colors   = COLOR_MAP.get(color_theme, COLOR_MAP["Indigo / Purple"])
    col      = colors["primary"]
    col_l    = colors["light"]
    font     = STYLE_FONTS.get(style, "'Inter',sans-serif")
    is_dark  = "Dark" in style or "Bold" in style
    bg       = "#0f1117" if is_dark else "#ffffff"
    bg2      = "#1a1f2e" if is_dark else "#f8f9fb"
    txt      = "#e2e8f0" if is_dark else "#1a202c"
    muted    = "#94a3b8" if is_dark else "#6b7280"
    card_bg  = "rgba(255,255,255,.04)" if is_dark else "#ffffff"
    card_brd = "rgba(255,255,255,.08)" if is_dark else "#e5e7eb"
    nav_bg   = "rgba(15,17,23,.92)" if is_dark else "rgba(255,255,255,.92)"

    hero_h, hero_p = HERO_MAP.get(wtype, ("Welcome to {brand}", "Building amazing experiences."))
    hero_h = hero_h.replace("{brand}", brand or title)

    services = SERVICE_MAP.get(wtype, SERVICE_MAP["Portfolio"])
    service_cards = "\n".join([
        f"""<div class="card"><div class="card-icon">{ic}</div>
        <h3>{t}</h3><p>{d}</p></div>"""
        for ic, t, d in services
    ])

    nav_links = {
        "Portfolio":"About|Work|Skills|Contact",
        "Business / Corporate":"Services|About|Team|Contact",
        "Landing Page":"Features|Pricing|FAQ|Get Started",
        "Education / School":"Courses|About|Team|Enroll",
        "Blog":"Posts|Topics|About|Contact",
        "E-Commerce":"Shop|Categories|About|Contact",
    }.get(wtype, "About|Services|Contact")
    nav_items = "".join(f"<li><a href='#'>{l}</a></li>" for l in nav_links.split("|"))

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1.0"/>
<title>{title}</title>
<meta name="description" content="{desc}"/>
<meta name="theme-color" content="{col}"/>
<meta property="og:title" content="{title}"/>
<meta name="author" content="HTML Quick-Styler"/>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&display=swap" rel="stylesheet"/>
<style>
,::before,*::after{{box-sizing:border-box;margin:0;padding:0}}
:root{{--col:{col};--bg:{bg};--bg2:{bg2};--txt:{txt};--muted:{muted};--card:{card_bg};--border:{card_brd}}}
html{{scroll-behavior:smooth}}
body{{font-family:{font};background:var(--bg);color:var(--txt);line-height:1.65;overflow-x:hidden}}
nav{{position:sticky;top:0;z-index:100;background:{nav_bg};backdrop-filter:blur(16px);
     border-bottom:1px solid var(--border);padding:.9rem 5%;
     display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:1rem}}
.logo{{font-size:1.2rem;font-weight:800;color:var(--col);text-decoration:none}}
nav ul{{display:flex;gap:1.8rem;list-style:none}}
nav a{{color:var(--muted);text-decoration:none;font-size:.875rem;font-weight:500;transition:color .2s}}
nav a:hover{{color:var(--col)}}
.nav-cta{{background:var(--col);color:#fff;padding:.5rem 1.2rem;border-radius:8px;
           font-weight:600;font-size:.85rem;text-decoration:none;transition:opacity .2s}}
.nav-cta:hover{{opacity:.85}}
/* HERO */
.hero{{min-height:88vh;display:flex;flex-direction:column;align-items:center;
       justify-content:center;text-align:center;padding:6rem 5% 4rem;position:relative;overflow:hidden}}
.hero::before{{content:'';position:absolute;top:-150px;left:50%;transform:translateX(-50%);
              width:700px;height:700px;
              background:radial-gradient(circle,{col}22 0%,transparent 65%);pointer-events:none}}
.hero-badge{{display:inline-flex;align-items:center;gap:.4rem;background:{col_l};
             border:1px solid {col}44;border-radius:999px;padding:.3rem .9rem;
             font-size:.78rem;font-weight:600;color:var(--col);margin-bottom:1.5rem}}
.hero h1{{font-size:clamp(2.4rem,5vw,4.5rem);font-weight:900;line-height:1.1;
          letter-spacing:-.03em;margin-bottom:1.2rem}}
.hero h1 span{{color:var(--col)}}
.hero p{{font-size:1.05rem;color:var(--muted);max-width:540px;margin:0 auto 2.2rem;line-height:1.75}}
.btn{{display:inline-flex;align-items:center;gap:.5rem;background:var(--col);color:#fff;
      padding:.85rem 2rem;border-radius:10px;font-weight:700;font-size:.95rem;
      text-decoration:none;transition:transform .25s,box-shadow .25s}}
.btn:hover{{transform:translateY(-3px);box-shadow:0 16px 40px {col}55}}
.btn-ghost{{background:transparent;color:var(--txt);border:1.5px solid var(--border);margin-left:.8rem}}
.btn-ghost:hover{{border-color:var(--col);color:var(--col);box-shadow:none}}
section{{padding:5rem 5%}}
.section-tag{{font-size:.75rem;font-weight:700;text-transform:uppercase;letter-spacing:.1em;
              color:var(--col);margin-bottom:.5rem;display:block}}
.section-title{{font-size:clamp(1.7rem,3vw,2.6rem);font-weight:800;line-height:1.2;margin-bottom:1rem}}
.section-sub{{font-size:1rem;color:var(--muted);max-width:540px;line-height:1.7}}
.text-center{{text-align:center}}.text-center .section-sub{{margin:0 auto 3rem}}
.grid-3{{display:grid;grid-template-columns:repeat(3,1fr);gap:1.2rem}}
.card{{background:var(--card);border:1px solid var(--border);border-radius:16px;padding:1.8rem;
       transition:transform .25s,box-shadow .25s,border-color .25s}}
.card:hover{{transform:translateY(-5px);box-shadow:0 20px 50px {col}22;border-color:{col}55}}
.card-icon{{font-size:1.8rem;margin-bottom:1rem}}
.card h3{{font-size:1rem;font-weight:700;margin-bottom:.5rem}}
.card p{{font-size:.875rem;color:var(--muted);line-height:1.6}}
/* ABOUT */
.about-grid{{display:grid;grid-template-columns:1fr 1fr;gap:4rem;align-items:center}}
.about-visual{{background:linear-gradient(135deg,{col},{col}88);border-radius:16px;
               aspect-ratio:1;display:flex;align-items:center;justify-content:center;font-size:5rem}}
.stats{{display:flex;gap:2rem;flex-wrap:wrap;margin-top:1.5rem}}
.stat-num{{font-size:2.2rem;font-weight:800;color:var(--col)}}
.stat-label{{font-size:.82rem;color:var(--muted)}}
/* TESTIMONIALS */
.testi-card{{position:relative}}
.testi-stars{{font-size:.9rem;margin-bottom:.8rem}}
.testi-text{{font-size:.875rem;color:var(--muted);line-height:1.7;font-style:italic;margin-bottom:1rem}}
.testi-author{{display:flex;align-items:center;gap:.7rem}}
.testi-av{{width:36px;height:36px;border-radius:50%;background:var(--col);
           display:flex;align-items:center;justify-content:center;font-size:.8rem;font-weight:700;color:#fff}}
.testi-name{{font-size:.875rem;font-weight:700}}
.testi-role{{font-size:.75rem;color:var(--muted)}}
/* CTA STRIP */
.cta-strip{{background:{col_l};border:1px solid {col}44;border-radius:24px;
            padding:4rem 3rem;text-align:center;margin:0 0 5rem}}
/* CONTACT */
.contact-wrap{{display:grid;grid-template-columns:1fr 1.2fr;gap:4rem;align-items:start}}
.form-g{{display:flex;flex-direction:column;gap:.4rem;margin-bottom:.9rem}}
.form-g label{{font-size:.8rem;font-weight:600;color:var(--muted)}}
.form-g input,.form-g textarea{{background:var(--card);border:1px solid var(--border);
  border-radius:8px;padding:.7rem .9rem;color:var(--txt);font-family:inherit;
  font-size:.875rem;outline:none;transition:border-color .2s;width:100%}}
.form-g input:focus,.form-g textarea:focus{{border-color:var(--col)}}
.form-g textarea{{min-height:110px;resize:vertical}}
.form-submit{{background:var(--col);color:#fff;border:none;padding:.85rem;border-radius:8px;
              font-weight:700;font-size:.95rem;cursor:pointer;width:100%;
              font-family:inherit;transition:opacity .2s}}
.form-submit:hover{{opacity:.85}}
footer{{background:{"#070a10" if is_dark else bg2};border-top:1px solid var(--border);
        padding:3rem 5% 1.5rem}}
.foot-top{{display:flex;justify-content:space-between;flex-wrap:wrap;gap:2rem;margin-bottom:2.5rem}}
.foot-brand p{{font-size:.82rem;color:var(--muted);margin-top:.5rem;max-width:220px}}
.foot-col h5{{font-size:.78rem;font-weight:700;text-transform:uppercase;letter-spacing:.08em;
              color:var(--muted);margin-bottom:.9rem}}
.foot-col ul{{list-style:none;display:flex;flex-direction:column;gap:.5rem}}
.foot-col a{{color:var(--muted);text-decoration:none;font-size:.85rem;transition:color .2s}}
.foot-col a:hover{{color:var(--col)}}
.foot-bottom{{border-top:1px solid var(--border);padding-top:1.5rem;
              font-size:.78rem;color:var(--muted);display:flex;justify-content:space-between;flex-wrap:wrap;gap:.5rem}}
@media(max-width:768px){{
  .grid-3,.about-grid,.contact-wrap,.foot-top{{grid-template-columns:1fr}}
  nav ul{{display:none}} .btn-ghost{{display:none}}
}}
.fade-up{{opacity:0;transform:translateY(20px);animation:fadeUp .6s ease forwards}}
@keyframes fadeUp{{to{{opacity:1;transform:translateY(0)}}}}
</style>
</head>
<body>

<nav>
  <a href="#" class="logo">{brand or title}</a>
  <ul>{nav_items}</ul>
  <a href="#contact" class="nav-cta">{cta}</a>
</nav>

<section class="hero">
  <div class="hero-badge">✦ {title}</div>
  <h1 class="fade-up">{hero_h.replace(brand or title, f'<span>{brand or title}</span>',1)}</h1>
  <p class="fade-up">{hero_p}</p>
  <div class="fade-up">
    <a href="#contact" class="btn">{cta} →</a>
    <a href="#services" class="btn btn-ghost">Learn More</a>
  </div>
</section>

<section id="services" style="background:var(--bg2)">
  <div class="text-center">
    <span class="section-tag">{"Skills & Expertise" if wtype=="Portfolio" else "Our Services"}</span>
    <h2 class="section-title">{"What I Do Best" if wtype=="Portfolio" else "What We Offer"}</h2>
    <p class="section-sub">{desc[:120]}{"..." if len(desc)>120 else ""}</p>
  </div>
  <div class="grid-3">{service_cards}</div>
</section>

<section id="about">
  <div class="about-grid">
    <div>
      <span class="section-tag">About {brand or title}</span>
      <h2 class="section-title">{"Crafting Digital Magic" if wtype=="Portfolio" else "Who We Are"}</h2>
      <p style="color:var(--muted);line-height:1.75;margin-bottom:1.5rem">{desc or "A passionate team building meaningful digital experiences that connect people and drive results."}</p>
      <div class="stats">
        <div><div class="stat-num">7+</div><div class="stat-label">Years Experience</div></div>
        <div><div class="stat-num">150+</div><div class="stat-label">Projects Done</div></div>
        <div><div class="stat-num">98%</div><div class="stat-label">Client Rate</div></div>
      </div>
    </div>
    <div class="about-visual">🚀</div>
  </div>
</section>

<section id="testimonials" style="background:var(--bg2)">
  <div class="text-center">
    <span class="section-tag">Testimonials</span>
    <h2 class="section-title">What Clients Say</h2>
    <p class="section-sub" style="margin:0 auto 3rem">Real feedback from real clients.</p>
  </div>
  <div class="grid-3">
    {"".join([f'''<div class="card testi-card"><div class="testi-stars">⭐⭐⭐⭐⭐</div>
    <p class="testi-text">"{t}"</p>
    <div class="testi-author"><div class="testi-av">{i}</div>
    <div><div class="testi-name">{n}</div><div class="testi-role">{r}</div></div></div></div>'''
    for i,n,r,t in [("S","Sarah K.","CEO, TechCorp","Absolutely exceptional work. Delivered beyond expectations."),
                     ("R","Raj M.","Founder, StartUp","The quality and attention to detail is unmatched."),
                     ("P","Priya S.","Marketing Head","Transformed our online presence completely!")]])}
  </div>
</section>

<div style="padding:0 5% 5rem">
  <div class="cta-strip">
    <span class="section-tag">Ready to begin?</span>
    <h2 class="section-title">{cta} — Let's Build Something Great</h2>
    <p style="color:var(--muted);margin-bottom:2rem">We'd love to hear about your project.</p>
    <a href="#contact" class="btn">{cta} →</a>
  </div>
</div>

<section id="contact" style="background:var(--bg2)">
  <div class="contact-wrap">
    <div>
      <span class="section-tag">Contact</span>
      <h2 class="section-title">Get In Touch</h2>
      <p style="color:var(--muted);margin-bottom:2rem;line-height:1.7">
        Have a project in mind? Let's talk. We respond within 24 hours.
      </p>
      <div style="display:flex;flex-direction:column;gap:.9rem">
        <div style="color:var(--muted);font-size:.9rem">📧 hello@{(brand or title).lower().replace(' ','')}.com</div>
        <div style="color:var(--muted);font-size:.9rem">📞 +91 98765 43210</div>
        <div style="color:var(--muted);font-size:.9rem">📍 Available Worldwide · Remote</div>
      </div>
    </div>
    <form onsubmit="document.getElementById('ok').style.display='block';this.reset();return false">
      <div class="form-g"><label>Name</label><input type="text" placeholder="Your name" required/></div>
      <div class="form-g"><label>Email</label><input type="email" placeholder="you@email.com" required/></div>
      <div class="form-g"><label>Message</label><textarea placeholder="Tell me about your project..."></textarea></div>
      <button type="submit" class="form-submit">{cta}</button>
      <div id="ok" style="display:none;margin-top:1rem;background:{col_l};border:1px solid {col}44;
           border-radius:8px;padding:.8rem;font-size:.85rem;color:var(--col)">
        ✓ Message sent! I'll be in touch soon.
      </div>
    </form>
  </div>
</section>

<footer>
  <div class="foot-top">
    <div class="foot-brand">
      <a class="logo" href="#" style="text-decoration:none">{brand or title}</a>
      <p>Crafting exceptional digital experiences with passion and precision.</p>
    </div>
    <div class="foot-col"><h5>Links</h5><ul>{nav_items}</ul></div>
    <div class="foot-col"><h5>Connect</h5><ul>
      <li><a href="#">LinkedIn</a></li><li><a href="#">GitHub</a></li><li><a href="#">Dribbble</a></li>
    </ul></div>
  </div>
  <div class="foot-bottom">
    <span>© 2026 {brand or title}. All rights reserved.</span>
    <span>Built with ⚡ HTML Quick Styler</span>
  </div>
</footer>

<script>
const obs=new IntersectionObserver(entries=>{{
  entries.forEach(e=>{{if(e.isIntersecting){{e.target.style.opacity='1';e.target.style.transform='translateY(0)';}}}})}},{{threshold:.1}});
document.querySelectorAll('.grid-3,.about-grid,.contact-wrap').forEach(el=>{{
  el.style.opacity='0';el.style.transform='translateY(24px)';
  el.style.transition='opacity .6s ease,transform .6s ease';obs.observe(el);
}});
</script>
</body>
</html>"""
    return html


# ─────────────────────────────────────────────────────────────────────────────
# IBM GRANITE GENERATION
# ─────────────────────────────────────────────────────────────────────────────
def granite_generate(title, desc, wtype, style, color_theme, cta, brand, max_tokens):
    prompt = f"""You are a senior web developer. Generate a complete professional HTML page.

Page Title: {title}
Brand / Author: {brand}
Website Type: {wtype}
Design Style: {style}
Color Theme: {color_theme}
CTA Button Text: {cta}
Page Sections: {desc}

Rules: Semantic HTML5, embedded CSS, Flexbox/Grid, hover effects, responsive.
Return ONLY the complete HTML starting with <!DOCTYPE html>:
<!DOCTYPE html>"""

    inputs  = tokenizer(prompt, return_tensors="pt").to(model.device)
    inp_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_tokens,
            do_sample=True, temperature=0.7, top_p=0.95,
            repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id,
        )
    raw  = tokenizer.decode(output[0][inp_len:], skip_special_tokens=True)
    html = "<!DOCTYPE html>" + raw
    end  = html.rfind("</html>")
    if end != -1:
        html = html[:end + 7]
    return html


# ─────────────────────────────────────────────────────────────────────────────
# MAIN HANDLER
# ─────────────────────────────────────────────────────────────────────────────
def generate_html(title, desc, wtype, style, color_theme, cta, brand, max_tokens):
    if not title.strip():
        gr.Warning("Please enter a Page Title.")
        return "", "<p style='color:red;padding:1rem;font-family:sans-serif'>⚠️ Please enter a Page Title first.</p>"

    if MODEL_LOADED:
        html_code = granite_generate(title, desc, wtype, style, color_theme, cta, brand, int(max_tokens))
    else:
        html_code = template_generate(title, desc, wtype, style, color_theme, cta, brand)

    return html_code, html_code


# ─────────────────────────────────────────────────────────────────────────────
# GRADIO UI
# ─────────────────────────────────────────────────────────────────────────────
WEBSITE_TYPES  = ["Portfolio","Business / Corporate","Landing Page","Education / School","Blog","E-Commerce","Restaurant / Café","Healthcare / Clinic"]
DESIGN_STYLES  = ["Modern","Minimal","Creative","Professional","Bold & Dark","Elegant"]
COLOR_THEMES   = list(COLOR_MAP.keys())
MODE_LABEL     = "🤖 IBM Granite 3.3 2B (GPU)" if MODEL_LOADED else "⚡ Template Engine (CPU)"
EXAMPLES = [
    ["Modern Portfolio Website","Portfolio","Modern","Indigo / Purple","Alex Johnson","Hero, skills, projects, testimonials, contact","Hire Me"],
    ["TechStart SaaS Landing","Landing Page","Bold & Dark","Blue / Ocean","TechStart","Hero, features, pricing, CTA, footer","Get Started Free"],
    ["Bright Minds Academy","Education / School","Professional","Green / Emerald","Bright Minds","Hero, courses, about, testimonials, contact","Enroll Now"],
]

css = """
.gen-btn{background:linear-gradient(135deg,#f0803c,#e05a1a)!important;color:white!important;font-weight:700!important}
.gen-btn:hover{opacity:.88!important}
footer{display:none!important}
"""

with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:

    gr.HTML(f"""
    <div style="background:#0d1117;padding:1rem 1.5rem;border-bottom:1px solid #30363d">
      <div style="color:#f0803c;font-weight:800;font-size:1.1rem">⚡ HTML Quick Styler</div>
      <div style="color:#8b949e;font-size:.8rem">SmartBridge Use-Case 2 &nbsp;·&nbsp; IBM Granite 3.3 2B Instruct + Gradio &nbsp;·&nbsp;
      Mode: <strong style="color:#10b981">{MODE_LABEL}</strong></div>
    </div>
    """)

    with gr.Tabs():

        with gr.TabItem("📄 Generate Page"):
            with gr.Row():
                with gr.Column(scale=1, min_width=320):
                    gr.Markdown("### Page Details")
                    page_title  = gr.Textbox(label="Page Title *", placeholder="e.g. Modern Portfolio Website")
                    brand_name  = gr.Textbox(label="Brand / Author Name", placeholder="e.g. Alex Johnson")
                    description = gr.Textbox(label="Sections & Description", lines=4,
                                             placeholder="Hero section, services grid, about, testimonials, contact form, footer")
                    with gr.Row():
                        web_type   = gr.Dropdown(WEBSITE_TYPES, value="Portfolio", label="Website Type")
                        des_style  = gr.Dropdown(DESIGN_STYLES, value="Modern", label="Design Style")
                    color_style = gr.Dropdown(COLOR_THEMES, value="Indigo / Purple", label="Color Style")
                    cta_text    = gr.Textbox(label="CTA Button Text", value="Get Started")
                    max_tok     = gr.Slider(512, 4096, value=2048, step=256, label="Max Tokens (GPU mode only)")
                    gen_btn     = gr.Button("✨ Generate HTML", elem_classes="gen-btn")
                    gr.Markdown("#### Quick Presets")
                    gr.Examples(
                        examples=EXAMPLES,
                        inputs=[page_title, web_type, des_style, color_style, brand_name, description, cta_text],
                    )

                with gr.Column(scale=2):
                    gr.Markdown("### Generated HTML / CSS")
                    html_out = gr.Code(label="HTML Output", language="html", lines=32, interactive=False)
                    clear_btn = gr.Button("🗑️ Clear")

        with gr.TabItem("👁️ Live Preview"):
            preview = gr.HTML(
                value="<div style='text-align:center;padding:6rem;color:#8b949e;font-family:sans-serif'>"
                      "🖥️ Generate a page first, then come back here to preview it.</div>"
            )

        with gr.TabItem("📖 Guide"):
            gr.Markdown(f"""
## How to Use HTML Quick Styler

*Current Mode:* {MODE_LABEL}

### Steps
1. Enter *Page Title* and *Brand Name*
2. Describe the *sections* you want (hero, services, about, contact, footer...)
3. Choose *Website Type, **Design Style, and **Color Theme*
4. Click *✨ Generate HTML*
5. Switch to *👁️ Live Preview* to see it rendered
6. Copy the code or save the file

### Technology Stack
| Component | Technology |
|---|---|
| AI Model | IBM Granite 3.3 2B Instruct |
| UI Framework | Gradio |
| Language | Python 3.11 |
| Output | HTML5 + CSS3 |
| Platform | Google Colab / Local |

> *Tip:* For full IBM Granite AI generation, run on *Google Colab with GPU* (free T4 GPU is sufficient).
""")

    # Wire up
    inputs_list  = [page_title, description, web_type, des_style, color_style, cta_text, brand_name, max_tok]
    outputs_list = [html_out, preview]
    gen_btn.click(fn=generate_html, inputs=inputs_list, outputs=outputs_list)
    clear_btn.click(
        fn=lambda: ("", "<div style='text-align:center;padding:4rem;color:#8b949e;font-family:sans-serif'>🗑️ Cleared</div>"),
        outputs=[html_out, preview]
    )


if __name__ == "__main__":
    print(f"\n Starting HTML Quick Styler [{MODE_LABEL}]...")
    demo.launch(share=False, server_name="127.0.0.1")

No GPU detected. Running in fast Template Engine mode.
For full IBM Granite AI generation, run on Google Colab with GPU.


/tmp/ipykernel_1498/3562974115.py:432: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_1498/3562974115.py:432: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:



 Starting HTML Quick Styler [⚡ Template Engine (CPU)]...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>